In [1]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
import re
from tqdm.auto import tqdm
import os

In [2]:
BASE_URL = (
    "https://www.cricbuzz.com/api/ipl-auction/completed/"
    "{cursor}/INR/0/ALL/ALL/0/ALL/apl.updated_at/DESC/ipl/{year}"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0 Safari/537.36"
    )
}

PRICE_RE = re.compile(r"\d+(?:\.\d+)?\s*(?:Cr|L)")

TEAMS = {
    "CSK","MI","RCB","KKR","RR","SRH",
    "DC","DD","PBKS","KXIP","GT","LSG",
    "GL","RPS","RPSG","PWI","KTK"
}

In [3]:
def fetch_completed_players(year, sleep=0.2):
    """
    Fetch all completed auction players for a given IPL auction year.

    Parameters
    ----------
    year : int or str
        Auction year (e.g. 2018)

    sleep : float
        Delay between API requests (seconds)

    Returns
    -------
    pandas.DataFrame
    """

    cursor = 0
    page = 1

    players = []
    seen = set()

    while True:

        url = BASE_URL.format(cursor=cursor, year=year)

        # Retry logic
        for attempt in range(5):
            try:
                response = requests.get(
                    url,
                    headers=HEADERS,
                    timeout=20
                )

                response.raise_for_status()

                response_json = response.json()

                break

            except Exception as e:

                if attempt == 4:
                    raise

                print(f"Retry {attempt+1}/5...")

                time.sleep(2)

        # End of pagination
        data = response_json.get("auctionPlayersList")

        if not data:
            break

        # Store players
        for p in data:

            pid = p["playerId"]

            if pid in seen:
                continue

            seen.add(pid)

            players.append({

                "id": p.get("id"),

                "playerId": pid,

                "playerName": p.get("playerName"),

                "country": p.get("country"),

                "countryId": p.get("countryId"),

                "role": p.get("role"),

                "season": p.get("season"),

                "basePrice": p.get("basePrice"),

                "auctionPrice": p.get("auctionPrice"),

                "auctionStatus": p.get("auctionStatus"),

                "playsForTeam": p.get("playsForTeam"),

                "updatedTime": int(p["updatedTime"]),

                "cappedStatus": p.get("cappedStatus"),

                "isPlayerOverseas": p.get(
                    "isPlayerOverseas",
                    False
                ),

                "playerURL": (
                    f"https://www.cricbuzz.com/cricket-series/"
                    f"ipl-{year}/auction/players/{pid}"
                )

            })

        # Next cursor
        new_cursor = data[-1]["updatedTime"]

        # Safety check
        if str(new_cursor) == str(cursor):
            print("Cursor did not advance. Stopping.")
            break

        cursor = new_cursor

        page += 1

        time.sleep(sleep)

    df = pd.DataFrame(players)

    return df

In [4]:
def _find_heading(soup, text):
    """Find a heading/div/span containing given text."""
    return soup.find(
        lambda tag:
            tag.name in ["h1", "h2", "h3", "h4", "div", "span"]
            and text.lower() in tag.get_text(" ", strip=True).lower()
    )

def extract_auction_trail(soup):

    bids = []

    for grid in soup.select("div.grid.rounded-\\[12px\\].grid-cols-\\[5fr\\,5fr\\]"):

        text = " ".join(grid.stripped_strings)

        price = PRICE_RE.search(text)
        if price is None:
            continue

        team = None
        for t in TEAMS:
            if f" {t} " in f" {text} ":
                team = t
                break

        if team is None:
            continue

        bids.append({
            "BidNumber": len(bids)+1,
            "Team": team,
            "BidAmount": price.group()
        })

    return pd.DataFrame(bids)


def extract_ipl_earnings(soup):
    earnings_heading = _find_heading(soup, "IPL Earnings")

    earnings_rows = []

    if earnings_heading:

        container = earnings_heading.parent

        grids = container.select("div.grid")

        for grid in grids:

            spans = [
                s.get_text(" ", strip=True)
                for s in grid.find_all("span")
            ]

            if len(spans) < 3:
                continue

            season = next(
                (
                    int(x)
                    for x in spans
                    if re.fullmatch(r"\d{4}", x)
                ),
                None
            )

            team = next(
                (
                    x for x in spans
                    if x.isupper()
                    and len(x) <= 4
                ),
                None
            )

            amount = next(
                (
                    x for x in spans
                    if PRICE_RE.fullmatch(x)
                ),
                None
            )

            if season is None or team is None or amount is None:
                continue

            earnings_rows.append({
                "Season": season,
                "Team": team,
                "Amount": amount
            })

    earnings_df = pd.DataFrame(earnings_rows)

    return earnings_df

def scrape_player_page(url):
    html = requests.get(url, headers=HEADERS).text
    soup = BeautifulSoup(html, "html.parser")

    trail_df = extract_auction_trail(soup)
    earnings_df = extract_ipl_earnings(soup)

    
    return trail_df , earnings_df

In [5]:
def scrape_all_player_pages(completed_players_df):
    """
    Scrape all Cricbuzz player auction pages.

    Parameters
    ----------
    completed_players_df : pandas.DataFrame
        Output of fetch_completed_players()

    Returns
    -------
    auction_trail_df : pandas.DataFrame
    earnings_df : pandas.DataFrame
    """

    all_trails = []
    all_earnings = []

    for _, row in tqdm(
        completed_players_df.iterrows(),
        total=len(completed_players_df)
    ):

        try:

            trail_df, earnings_df = scrape_player_page(row["playerURL"])

            player_info = row.to_dict()

            if not trail_df.empty:
                trail_df = trail_df.assign(**player_info)
                all_trails.append(trail_df)

            if not earnings_df.empty:
                earnings_df = earnings_df.assign(**player_info)
                all_earnings.append(earnings_df)

        except Exception as e:

            print(f"✗ {row['playerName']} :: {e}")

    auction_trail_df = (
        pd.concat(all_trails, ignore_index=True)
        if all_trails
        else pd.DataFrame()
    )

    earnings_df = (
        pd.concat(all_earnings, ignore_index=True)
        if all_earnings
        else pd.DataFrame()
    )

    return auction_trail_df, earnings_df

In [6]:

os.makedirs("completed_players", exist_ok=True)
os.makedirs("auction_trail", exist_ok=True)
os.makedirs("earnings", exist_ok=True)

for year in range(2018, 2027):
    completed_players_df = fetch_completed_players(year)

    print(f"Scraping Year {year} ...")
    auction_trail_df, earnings_df = scrape_all_player_pages(completed_players_df)

    print(
        completed_players_df.shape,
        auction_trail_df.shape,
        earnings_df.shape
    )

    completed_players_df.to_csv(
        f"completed_players/completed_players_{year}.csv",
        index=False
    )

    auction_trail_df.to_csv(
        f"auction_trail/auction_trail_{year}.csv",
        index=False
    )

    earnings_df.to_csv(
        f"earnings/earnings_{year}.csv",
        index=False
    )

Scraping Year 2018 ...


  0%|          | 0/300 [00:00<?, ?it/s]

(300, 15) (2406, 18) (997, 18)
Scraping Year 2019 ...


  0%|          | 0/270 [00:00<?, ?it/s]

(270, 15) (665, 18) (1109, 18)
Scraping Year 2020 ...


...

(307, 15) (1159, 18) (1502, 18)
Scraping Year 2025 ...


  0%|          | 0/402 [00:00<?, ?it/s]

(402, 15) (2630, 18) (1719, 18)
Scraping Year 2026 ...


  0%|          | 0/329 [00:00<?, ?it/s]

(329, 15) (968, 18) (1605, 18)


In [7]:
auction_trail_df

,BidNumber,Team,BidAmount,id,playerId,playerName,country,countryId,role,season,basePrice,auctionPrice,auctionStatus,playsForTeam,updatedTime,cappedStatus,isPlayerOverseas,playerURL
0,1,DC,2.00 Cr,33084,9441,Kyle Jamieson,New Zealand,13,Bowler,ipl-2026,2.00 Cr,2.00 Cr,sold,DC,1765899655,CAPPED,True,https://www.cricbuzz.com/cricket-series/ipl-20...
1,1,RCB,30.00 L,35395,1457033,Kanishk Chouhan,India,2,Allrounder,ipl-2026,30.00 L,30.00 L,sold,RCB,1765899602,UNCAPPED,False,https://www.cricbuzz.com/cricket-series/ipl-20...
2,1,RCB,30.00 L,35252,1430529,Vihaan Malhotra,India,2,Bowler,ipl-2026,30.00 L,30.00 L,sold,RCB,1765899573,UNCAPPED,False,https://www.cricbuzz.com/cricket-series/ipl-20...
3,1,GT,75.00 L,33678,9821,Luke Wood,England,9,Bowler,ipl-2026,75.00 L,75.00 L,sold,GT,1765899519,CAPPED,True,https://www.cricbuzz.com/cricket-series/ipl-20...
4,1,GT,30.00 L,34393,13113,Prithvi Raj Yarra,India,2,Bowler,ipl-2026,30.00 L,30.00 L,sold,GT,1765899466,UNCAPPED,False,https://www.cricbuzz.com/cricket-series/ipl-20...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963,114,RR,2.60 Cr,32864,12225,Cameron Green,Australia,4,Batter,ipl-2026,2.00 Cr,25.20 Cr,sold,KKR,1765877226,CAPPED,True,https://www.cricbuzz.com/cricket-series/ipl-20...
964,115,MI,2.40 Cr,32864,12225,Cameron Green,Australia,4,Batter,ipl-2026,2.00 Cr,25.20 Cr,sold,KKR,1765877226,CAPPED,True,https://www.cricbuzz.com/cricket-series/ipl-20...
965,116,RR,2.20 Cr,32864,12225,Cameron Green,Australia,4,Batter,ipl-2026,2.00 Cr,25.20 Cr,sold,KKR,1765877226,CAPPED,True,https://www.cricbuzz.com/cricket-series/ipl-20...
966,117,MI,2.00 Cr,32864,12225,Cameron Green,Australia,4,Batter,ipl-2026,2.00 Cr,25.20 Cr,sold,KKR,1765877226,CAPPED,True,https://www.cricbuzz.com/cricket-series/ipl-20...


In [8]:
trail_df, earnings_df = scrape_player_page("https://www.cricbuzz.com/cricket-series/ipl-2018/auction/players/6557")

In [9]:
trail_df.sort_values(by="BidAmount")

,BidNumber,Team,BidAmount
14,15,KKR,10.00 Cr
13,14,KXIP,10.50 Cr
12,13,KKR,11.00 Cr
0,1,RR,12.50 Cr
33,34,CSK,2.00 Cr
30,31,KXIP,2.20 Cr
32,33,CSK,2.40 Cr
27,28,KXIP,2.60 Cr
35,36,CSK,2.80 Cr
21,22,KXIP,3.00 Cr
